# 실행 환경 확인

워크스페이스의 파이썬, CUDA, GPU가 정상인지 확인하는 노트북입니다. 위에서부터 차례로 실행하세요.

## 1. 파이썬과 시스템

In [1]:
import os
import platform
import sys

# 세 프레임워크가 한 GPU를 나눠 쓰도록 메모리 선점을 끈다 (이미지에 이미 설정돼 있으면 그대로 둔다)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
os.environ.setdefault("KERAS_BACKEND", "jax")

print("Python  :", sys.version)
print("실행 파일:", sys.executable)
print("OS      :", platform.platform())
print("CPU 코어:", os.cpu_count())

Python  : 3.12.16 (main, Nov  5 2026, 07:58:20) [GCC 14.2.0]
실행 파일: /usr/local/bin/python
OS      : Linux-5.15.0-86-generic-x86_64-with-glibc2.41
CPU 코어: 16


## 2. GPU와 드라이버

`nvidia-smi`가 실패하면 컨테이너에 GPU가 할당되지 않은 것입니다.

In [2]:
!nvidia-smi

[Coaster NOTICE (pid:10015 thread=140017538881408 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
Mon Nov 23 06:42:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 620.70.06              KMD Version: 620.70.06     CUDA UMD Version: 13.5     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  PART_8 NVIDIA RTX A2000        On  |   00000000:03:00.0 Off |                  Off |
| 30%   24C    P8             12W /   70W |       0MiB /    

## 3. 설치된 패키지 버전

In [2]:
from importlib.metadata import PackageNotFoundError, version

packages = [
    "tensorflow", "torch", "jax", "jaxlib",
    "keras", "keras-hub", "keras-tuner",
    "numpy", "matplotlib", "jupyterlab",
]
for name in packages:
    try:
        print(f"{name:12s} {version(name)}")
    except PackageNotFoundError:
        print(f"{name:12s} (설치되지 않음)")

tensorflow   2.21.0
torch        2.14.1+cu126
jax          0.12.1
jaxlib       0.12.1
keras        3.15.1
keras-hub    0.33.0
keras-tuner  1.4.8
numpy        2.5.5
matplotlib   3.11.2
jupyterlab   4.6.6


## 4. PyTorch

In [1]:
import torch

print("torch       :", torch.__version__)
print("CUDA (빌드) :", torch.version.cuda)
print("cuDNN       :", torch.backends.cudnn.version())
print("GPU 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}, {props.total_memory / 1024**3:.1f} GB")
    x = torch.rand(2000, 2000, device="cuda")
    print("행렬 곱 결과 장치:", (x @ x).device)

[Coaster NOTICE (pid:3894 thread=139820515862400 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)


torch       : 2.14.1+cu126
CUDA (빌드) : 12.6
cuDNN       : 91002
GPU 사용 가능: True
GPU 0: NVIDIA RTX A2000, 0.5 GB


[Coaster NOTICE (pid:3894 thread=139820515862400 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
[Coaster NOTICE (pid:3894 thread=139820515862400 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
[Coaster NOTICE (pid:3894 thread=139820515862400 comm.c:387)]: block_container_agent: resolve_host_pid response: host_pid=3504423
[Coaster NOTICE (pid:3894 thread=139820515862400 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
[Coaster NOTICE (pid:3894 thread=139820515862400 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
[Coaster NOTICE (pid:3894 thread=139820515862400 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
[Coaster ERROR 

행렬 곱 결과 장치: cuda:0


## 5. TensorFlow

In [2]:
import tensorflow as tf

build = tf.sysconfig.get_build_info()
print("tensorflow  :", tf.__version__)
print("CUDA (빌드) :", build.get("cuda_version"))
print("cuDNN (빌드):", build.get("cudnn_version"))

gpus = tf.config.list_physical_devices("GPU")
print("GPU 목록    :", gpus)

if gpus:
    x = tf.random.uniform((2000, 2000))
    print("행렬 곱 결과 장치:", tf.matmul(x, x).device)

I0000 00:00:1795445397.573636    3894 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


tensorflow  : 2.21.0
CUDA (빌드) : 12.5.1
cuDNN (빌드): 9
GPU 목록    : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
행렬 곱 결과 장치: /job:localhost/replica:0/task:0/device:GPU:0


[Coaster NOTICE (pid:3894 thread=139820515862400 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
I0000 00:00:1795445400.117495    3894 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 237 MB memory:  -> device: 0, name: NVIDIA RTX A2000, pci bus id: 0000:03:00.0, compute capability: 8.6


## 6. JAX

In [6]:
import jax
import jax.numpy as jnp

print("jax      :", jax.__version__)
print("장치 목록:", jax.devices())

x = jax.random.uniform(jax.random.key(0), (2000, 2000))
print("행렬 곱 결과 장치:", (x @ x).devices())

jax      : 0.12.1
장치 목록: [CudaDevice(id=0)]


[Coaster NOTICE (pid:9943 thread=139776553387904 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
[Coaster NOTICE (pid:9943 thread=139776553387904 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
E1123 06:47:08.025687   10514 cuda_executor.cc:1552] [0] Failed to allocate device memory of 128.00MiB (134217728 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
=== Source Location Trace: ===
external/xla+/xla/stream_executor/cuda/cuda_status.cc:45
external/xla+/xla/stream_executor/cuda/cuda_device_allocator.cc:356
external/xla+/xla/stream_executor/cuda/cuda_device_allocator.cc:676

E1123 06:47:08.025898   10514 cuda_executor.cc:1552] [0] Failed to allocate device memory of 115.20MiB (120795904 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
=== Source Location Trace: ===
external/xla+/xla/stream_executor/

행렬 곱 결과 장치: {CudaDevice(id=0)}


## 7. 케라스로 간단한 모델 훈련

무작위 데이터로 작은 모델을 한 번 훈련해 전체 흐름이 동작하는지 확인합니다.

In [1]:
import keras
import numpy as np

print("keras  :", keras.__version__)
print("백엔드 :", keras.backend.backend())

x_train = np.random.rand(2000, 20).astype("float32")
y_train = (x_train.sum(axis=1) > 10).astype("float32")

model = keras.Sequential([
    keras.Input(shape=(20,)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history = model.fit(x_train, y_train, epochs=5, batch_size=64, verbose=2)

I0000 00:00:1795418737.183357     828 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


keras  : 3.15.1
백엔드 : tensorflow


[Coaster NOTICE (pid:828 thread=140505550112640 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
[Coaster NOTICE (pid:828 thread=140505550112640 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
[Coaster NOTICE (pid:828 thread=140505550112640 comm.c:387)]: block_container_agent: resolve_host_pid response: host_pid=3420360
[Coaster NOTICE (pid:828 thread=140505550112640 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
I0000 00:00:1795418739.796484     828 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 141 MB memory:  -> device: 0, name: NVIDIA RTX A2000, pci bus id: 0000:03:00.0, compute capability: 8.6


Epoch 1/5


I0000 00:00:1795418741.586291     899 service.cc:153] XLA service 0x7fc820030f80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1795418741.586353     899 service.cc:161]   StreamExecutor [0]: NVIDIA RTX A2000, Compute Capability 8.6 (Driver: 13.5.0; Runtime: 12.6.0; Toolkit: 12.5.0; DNN: 9.10.2)
I0000 00:00:1795418741.620471     899 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
[Coaster NOTICE (pid:828 thread=140498161358528 utils.c:161)]: load_gpc_group: no GPC entries in /etc/block_gpu/allocation/gpc_group (pre-Hopper or empty file, skipping)
I0000 00:00:1795418741.792647     899 cuda_dnn.cc:461] Loaded cuDNN version 91002
I0000 00:00:1795418741.805037     899 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1054__.8
I0000 00:00:1795418741.816443     899 dot_search_space.cc:240] All configs were filtered out because none of them sufficien

32/32 - 8s - 251ms/step - accuracy: 0.4930 - loss: 0.7136
Epoch 2/5
32/32 - 0s - 9ms/step - accuracy: 0.5325 - loss: 0.6870
Epoch 3/5
32/32 - 0s - 7ms/step - accuracy: 0.5755 - loss: 0.6770
Epoch 4/5
32/32 - 0s - 9ms/step - accuracy: 0.5690 - loss: 0.6687
Epoch 5/5
32/32 - 0s - 6ms/step - accuracy: 0.6005 - loss: 0.6572


In [3]:
  import tensorflow as tf
  print(tf.config.list_physical_devices("GPU"))


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
